# Complete DNN Training Pipeline on Google Colab

This notebook performs the entire pipeline:
1. Generates reference trajectory
2. Builds MPC controller
3. Collects training data from MPC (10,000 anchors)
4. Trains DNN to mimic MPC
5. Evaluates the trained model

## Instructions:
1. Set Runtime → Change runtime type → GPU (T4 or better)
2. Run all cells sequentially
3. Download the trained model when complete

In [ ]:
# Mount Google Drive (optional - for saving files)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies
!pip install torch torchvision torchaudio numpy matplotlib scipy tqdm casadi
!pip install opengen

In [ ]:
# Setup project structure
import os
import sys
import numpy as np
import torch
from pathlib import Path

# Create directories
!mkdir -p src/controllers src/utils src/training models data outputs my_optimizers

# Create __init__ files
for d in ['src', 'src/controllers', 'src/utils', 'src/training']:
    Path(f'{d}/__init__.py').touch()

print("Project structure created")

In [ ]:
# Upload source code files
from google.colab import files
print("Upload the following source files:")
print("  - src/controllers/mpc_controller.py")
print("  - src/controllers/dnn_controller.py")
print("  - src/utils/reference.py")
print("  - src/utils/visualization.py")
print("  - src/utils/data_collection.py")
print("  - src/training/trainer.py")
uploaded = files.upload()

In [ ]:
# Setup reference trajectory
sys.path.insert(0, 'src')
from utils.reference import generate_ellipse_reference

total_points = 10000
t_param, x_ref_array, y_ref_array, theta_ref_array = generate_ellipse_reference(
    total_points=total_points, height=4.0, width=0.2
)

print(f"Reference trajectory created: {len(x_ref_array)} points")

In [ ]:
# Build MPC controller
from controllers.mpc_controller import MPCController

# Define MPC parameters
nu = 3  # [s, omega, v]
nx_aug = 4  # [x, y, theta, theta_p]
N = 70
ts = 0.03
q = 100
qtheta = 10
r_control = 0.1
r_v = 0.001
v_ref = 0.2

print("Building MPC optimizer...")
mpc = MPCController(nu=nu, nx_aug=nx_aug, N=N, ts=ts,
                   q=q, qtheta=qtheta, r_control=r_control, r_v=r_v, v_ref=v_ref)
solver = mpc.build_optimizer(build_dir="my_optimizers")
print("MPC optimizer ready!")

In [ ]:
# Collect training data from MPC
from utils.data_collection import MPCDataCollector

collector = MPCDataCollector(solver, t_param, x_ref_array, y_ref_array, 
                             theta_ref_array, N, nu, ts, v_ref)

print("Collecting training data (this will take a while)...")
states, references, controls, failure_info, block_ids = collector.collect_data_corridor(
    num_anchors=10000, offset_distance=0.1, 
    heading_error_range=np.pi/6, keep_failures=True
)

print(f"Successfully collected {len(states)} samples")

In [ ]:
# Save dataset
np.savez('data/mpc_dataset.npz',
         states=states,
         references=references,
         controls=controls,
         block_ids=block_ids,
         x_ref_array=x_ref_array,
         y_ref_array=y_ref_array,
         t_param=t_param,
         theta_ref_array=theta_ref_array,
         N=N, nu=nu, ts=ts)
print("Dataset saved to 'data/mpc_dataset.npz'")

In [ ]:
# Prepare training data
from training.trainer import prepare_dataloaders

batch_size = 256 if torch.cuda.is_available() else 32
print(f"Using batch_size={batch_size} ({'GPU' if torch.cuda.is_available() else 'CPU'})")

train_loader, val_loader, dataset = prepare_dataloaders(
    references, controls, train_split=0.8, batch_size=batch_size,
    block_ids=block_ids, standardize_targets=True
)
print(f"Training: {len(train_loader.dataset)} samples | Validation: {len(val_loader.dataset)} samples")

In [ ]:
# Create model
from controllers.dnn_controller import MPCNeuralNetwork

ref_feature_dim = references.shape[1]
print(f"Network input: references({ref_feature_dim}) dims")

model = MPCNeuralNetwork(
    ref_feature_dim=ref_feature_dim,
    hidden_dims=[1024, 1024, 512, 256, 128],
    output_dim=2
)

num_params = sum(p.numel() for p in model.parameters())
print(f"Model created with {num_params:,} parameters")

In [ ]:
# Train the model
from training.trainer import train_model

print("Training model (with increased patience=150, scheduler patience=30)...")
trained_model = train_model(
    model, train_loader, val_loader, references,
    epochs=1000, lr=0.001, seed=42, batch_size=batch_size
)

In [ ]:
# Save the model
torch.save(trained_model.state_dict(), 'models/final_mpc_dnn_model.pth')
print("Model saved as 'models/final_mpc_dnn_model.pth'")

In [ ]:
# Download the trained model
from google.colab import files
files.download('models/final_mpc_dnn_model.pth')

In [ ]:
# Optional: Save to Google Drive
!cp models/final_mpc_dnn_model.pth /content/drive/MyDrive/
print("Model saved to Google Drive")

In [ ]:
# Optional: Download the dataset as well
files.download('data/mpc_dataset.npz')

## Summary

This notebook performed the complete training pipeline:

1. ✓ Generated reference trajectory (10,000 points)
2. ✓ Built MPC controller
3. ✓ Collected training data (10,000 anchors → ~80,000 samples)
4. ✓ Trained DNN with increased patience (150 epochs) and scheduler patience (30)
5. ✓ Saved trained model

**Files to download:**
- `models/final_mpc_dnn_model.pth` - Trained DNN model
- `data/mpc_dataset.npz` - Training dataset (optional)

**Changes made:**
- Early stopping patience: 40 → 150 epochs
- Learning rate scheduler patience: 15 → 30 epochs
- Dataset size: 5,000 → 10,000 anchors

These changes should result in better DNN performance with more training data and longer training time.